In [1]:
import os
import time
from datetime import datetime
from pyspark import StorageLevel
import pyspark

_home = os.environ.get("SPARK_HOME", "")
if not os.path.exists(os.path.join(_home, "bin", "spark-submit")):
    _pkg = os.path.dirname(pyspark.__file__)
    if os.path.exists(os.path.join(_pkg, "bin", "spark-submit")):
        os.environ["SPARK_HOME"] = _pkg
os.environ.pop("PYSPARK_SUBMIT_ARGS", None)

from pyspark.sql import SparkSession, Window, functions as F

In [3]:
LAKE = os.environ.get("LAKE_PATH", "file:///data-lake/data-lake")
BRONZE, SILVER, GOLD = LAKE + "/bronze", LAKE + "/silver", LAKE + "/gold"
REJECTED = SILVER + "/_rejected"
RUN_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

CORES = os.environ.get("SILVER_CORES", "2")
MEM = os.environ.get("SILVER_MEM", "2g")
spark = (
    SparkSession.builder.master("local[%s]" % CORES).appName("gold")
    .config("spark.driver.memory", MEM)
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.hadoop.parquet.block.size", str(32 * 1024 * 1024))
    .getOrCreate()
)
print("Spark %s | master=%s | driver memory=%s"
      % (spark.version, spark.sparkContext.master, spark.sparkContext.getConf().get("spark.driver.memory", "default (1g)")))
c = F.col

def silver(table):
    return spark.read.parquet(SILVER + "/" + table)

Spark 3.1.2 | master=local[2] | driver memory=2g


In [5]:
# ---------------------------------------------------------------------------
# 1) the join of 4 datasets (the dimensions are small -> broadcast join, no shuffle of the big table)
# ---------------------------------------------------------------------------
fact = silver("FactOnlineSales")
prod = silver("DimProduct").select("ProductKey", "ProductName", "BrandName", "ClassName")
cust = silver("DimCustomer").select("CustomerKey", "FullName", "CustomerType")
store = silver("DimStore").select("StoreKey", "StoreName", "StoreType")

for name, df, key in [
    ("DimProduct", prod, "ProductKey"),
    ("DimCustomer", cust, "CustomerKey"),
    ("DimStore", store, "StoreKey"),
]:
    duplicate_keys = (
        df.groupBy(key)
        .count()
        .filter(c("count") > 1)
        .count()
    )

    null_keys = df.filter(c(key).isNull()).count()

    print(
        "%s: duplicate_keys=%d null_keys=%d"
        % (name, duplicate_keys, null_keys)
    )

    if duplicate_keys > 0 or null_keys > 0:
        raise ValueError(
            "%s contains duplicate or null join keys" % name
        )
    
enriched = (
    fact.join(F.broadcast(prod), "ProductKey", "left")
        .join(F.broadcast(cust), "CustomerKey", "left")
        .join(F.broadcast(store), "StoreKey", "left")
        .fillna({"ProductName": "Unknown", "BrandName": "Unknown", "ClassName": "Unknown",
                 "FullName": "Unknown", "CustomerType": "Unknown", "StoreName": "Unknown", "StoreType": "Unknown"})
        .withColumn("NetAfterReturns", c("SalesAmount") - c("ReturnAmount"))      # same formulas as the DWH
        .withColumn("Profit", c("SalesAmount") - c("TotalCost"))
)
enriched.createOrReplaceTempView("sales_enriched")


DimProduct: duplicate_keys=0 null_keys=0
DimCustomer: duplicate_keys=0 null_keys=0
DimStore: duplicate_keys=0 null_keys=0


In [6]:
# ---------------------------------------------------------------------------
# 2) Gold datasets with DataFrame aggregations
# ---------------------------------------------------------------------------
top_products = (
    enriched.groupBy("ProductKey", "ProductName", "BrandName", "ClassName")
    .agg(F.sum("SalesQuantity").alias("units_sold"),
         F.sum("SalesAmount").alias("revenue"),
         F.sum("NetAfterReturns").alias("net_revenue"),
         F.sum("Profit").alias("profit"),
         F.countDistinct("SalesOrderNumber").alias("orders"))
    .withColumn("revenue_rank", F.dense_rank().over(Window.orderBy(c("revenue").desc())))
)

customer_frequency = (
    enriched.groupBy("CustomerKey", "FullName", "CustomerType")
    .agg(F.countDistinct("SalesOrderNumber").alias("orders"),
         F.count("*").alias("order_lines"),
         F.sum("SalesAmount").alias("total_spent"),
         F.min("SaleDate").alias("first_purchase"),
         F.max("SaleDate").alias("last_purchase"))
    .withColumn("avg_order_value", c("total_spent") / c("orders"))
    .withColumn("frequency_band", F.when(c("orders") >= 5, "Loyal").when(c("orders") >= 2, "Repeat").otherwise("One-time"))
)

monthly_brand = (
    enriched.groupBy("SaleYear", "SaleMonth", "BrandName")
    .agg(F.sum("SalesQuantity").alias("units_sold"),
         F.sum("SalesAmount").alias("revenue"),
         F.sum("Profit").alias("profit"),
         F.countDistinct("SalesOrderNumber").alias("orders"))
)

store_monthly = (
    enriched.groupBy("SaleYear", "SaleMonth", "StoreKey", "StoreName")
    .agg(F.sum("SalesAmount").alias("revenue"),
         F.sum("ReturnAmount").alias("returns_amount"),
         F.countDistinct("SalesOrderNumber").alias("orders"))
    .withColumn(
    "return_rate_pct",
    F.when(
        c("revenue").isNull() | (c("revenue") <= 0),
        F.lit(None).cast("double")
    ).otherwise(F.round(100 * c("returns_amount") / c("revenue"), 2)))
)

In [7]:
# ---------------------------------------------------------------------------
# 3) Gold datasets with Spark SQL
# ---------------------------------------------------------------------------
sql_top_customers = spark.sql("""
    SELECT CustomerKey, FullName, CustomerType,
           COUNT(DISTINCT SalesOrderNumber) AS orders,
           SUM(SalesAmount)                 AS total_spent
    FROM sales_enriched
    GROUP BY CustomerKey, FullName, CustomerType
    ORDER BY total_spent DESC
    LIMIT 10
""")

sql_brand_share = spark.sql("""
    SELECT SaleYear, BrandName,
           SUM(SalesAmount) AS revenue,
           ROUND(100 * SUM(SalesAmount) / SUM(SUM(SalesAmount)) OVER (PARTITION BY SaleYear), 2) AS pct_of_year
    FROM sales_enriched
    GROUP BY SaleYear, BrandName
""")

# name, dataframe, partition columns, the business question it answers
DATASETS = [
    ("top_products", top_products, None,
     "Which products sell the most (units, revenue, profit) and how do they rank?"),
    ("customer_purchase_frequency", customer_frequency, None,
     "How often does each customer buy and how much do they spend (Loyal / Repeat / One-time)?"),
    ("monthly_sales_by_brand", monthly_brand, ["SaleYear"],
     "How do revenue, units and profit move month by month for each brand?"),
    ("store_monthly_kpi", store_monthly, ["SaleYear"],
     "How does each store perform every month (revenue, orders, return rate)?"),
    ("sql_top_customers", sql_top_customers, None,
     "(Spark SQL) Who are the 10 customers with the highest spend?"),
    ("sql_brand_share_by_year", sql_brand_share, None,
     "(Spark SQL) What share of each year's revenue does every brand bring?"),
]

results = []
for name, df, part, question in DATASETS:
    t0 = time.time()
    path = GOLD + "/" + name
    writer = df.write.mode("overwrite")
    if part:
        writer = writer.partitionBy(*part)
    writer.parquet(path)
    rows = spark.read.parquet(path).count()
    seconds = round(time.time() - t0, 1)
    print("gold/%-30s rows=%10d  %6.1fs" % (name, rows, seconds))
    results.append((name, question, int(rows), float(seconds), RUN_AT))


gold/top_products                   rows=      2516    46.8s
gold/customer_purchase_frequency    rows=     18869    40.1s
gold/monthly_sales_by_brand         rows=       396    49.7s
gold/store_monthly_kpi              rows=       108    48.5s
gold/sql_top_customers              rows=        10    40.0s
gold/sql_brand_share_by_year        rows=        33    31.0s


In [8]:
# ---------------------------------------------------------------------------
# 4) checks
# ---------------------------------------------------------------------------
print("\n--- checks")
fact_rows, enriched_rows = fact.count(), enriched.count()
print("join keeps the row count : fact=%d enriched=%d  %s"
      % (fact_rows, enriched_rows, "OK" if fact_rows == enriched_rows else "MISMATCH"))

silver_total = fact.agg(F.sum("SalesAmount")).first()[0]
gold_total = top_products.agg(F.sum("revenue")).first()[0]
print("silver total = gold total: %s vs %s  %s" % (silver_total, gold_total, "OK" if silver_total == gold_total else "MISMATCH"))

bronze = spark.read.parquet(BRONZE + "/FactOnlineSales")
bronze_total = bronze.agg(F.sum("SalesAmount")).first()[0]
rejected = spark.read.parquet(
    REJECTED + "/FactOnlineSales"
)

rejected_keys = (rejected.filter(c("RecordID") != "(null key)").select(c("RecordID").cast("int").alias("OnlineSalesKey")).distinct())
rejected_total = (bronze.join(F.broadcast(rejected_keys), "OnlineSalesKey", "inner").agg(F.sum("SalesAmount")).first()[0])
rejected_total = rejected_total if rejected_total is not None else 0
print("bronze = silver + rejected: %s = %s + %s  %s"
      % (bronze_total, silver_total, rejected_total, "OK" if bronze_total == silver_total + rejected_total else "MISMATCH"))
 


--- checks
join keeps the row count : fact=12527442 enriched=12527442  OK
silver total = gold total: 2718202629.8082 vs 2718202629.8082  OK
bronze = silver + rejected: 2718202629.8082 = 2718202629.8082 + 0.0000  OK


In [9]:
# ---------------------------------------------------------------------------
# 5) monitoring tables
# ---------------------------------------------------------------------------
spark.createDataFrame(
    results, "dataset string, business_question string, rows long, seconds double, run_at string"
).write.mode("overwrite").parquet(LAKE + "/monitoring/gold_datasets")

counts = []
for layer, base, names in [("bronze", BRONZE, ["FactOnlineSales", "DimProduct", "DimCustomer", "DimStore"]),
                           ("silver", SILVER, ["FactOnlineSales", "DimProduct", "DimCustomer", "DimStore"]),
                           ("gold", GOLD, [d[0] for d in DATASETS])]:
    for n in names:
        counts.append((layer, n, int(spark.read.parquet(base + "/" + n).count()), RUN_AT))
lake_counts = spark.createDataFrame(counts, "layer string, dataset string, rows long, run_at string")
lake_counts.write.mode("overwrite").parquet(LAKE + "/monitoring/lake_row_counts")
print("\n--- rows per layer")
lake_counts.show(40, truncate=False)
spark.stop()


--- rows per layer
+------+---------------------------+--------+-------------------+
|layer |dataset                    |rows    |run_at             |
+------+---------------------------+--------+-------------------+
|bronze|FactOnlineSales            |12627608|2026-10-27 21:27:54|
|bronze|DimProduct                 |2517    |2026-10-27 21:27:54|
|bronze|DimCustomer                |18869   |2026-10-27 21:27:54|
|bronze|DimStore                   |306     |2026-10-27 21:27:54|
|silver|FactOnlineSales            |12527442|2026-10-27 21:27:54|
|silver|DimProduct                 |2517    |2026-10-27 21:27:54|
|silver|DimCustomer                |18869   |2026-10-27 21:27:54|
|silver|DimStore                   |306     |2026-10-27 21:27:54|
|gold  |top_products               |2516    |2026-10-27 21:27:54|
|gold  |customer_purchase_frequency|18869   |2026-10-27 21:27:54|
|gold  |monthly_sales_by_brand     |396     |2026-10-27 21:27:54|
|gold  |store_monthly_kpi          |108     |2026-10-27 